## Google Meet Transcription

In [4]:
# %pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [5]:
import os
import google.auth
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# Scopes required for accessing Google Calendar and Drive APIs
SCOPES = ['https://www.googleapis.com/auth/calendar.readonly',
          'https://www.googleapis.com/auth/drive.readonly']

def authenticate_google_services():
    """
    Authenticate and return the Google Calendar and Drive API services.

    Uses OAuth 2.0 to authorize access to Google Calendar and Google Drive APIs.
    Saves and reloads credentials from 'token.json' for subsequent requests.

    Returns:
        tuple: Contains two Google API service objects:
            - calendar_service (googleapiclient.discovery.Resource): Service for Google Calendar API.
            - drive_service (googleapiclient.discovery.Resource): Service for Google Drive API.
    """
    creds = None
    # Load credentials from token file if it exists
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    # If credentials are not available or invalid, get new ones
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the new credentials for future use
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    
    # Build the Google Calendar and Drive services
    calendar_service = build('calendar', 'v3', credentials=creds)
    drive_service = build('drive', 'v3', credentials=creds)
    return calendar_service, drive_service

def get_meeting_metadata(calendar_service, calendar_id='primary', event_id=''):
    """
    Retrieve metadata for a specific event from Google Calendar.

    Args:
        calendar_service (googleapiclient.discovery.Resource): Google Calendar API service object.
        calendar_id (str): The ID of the calendar to query (default is 'primary').
        event_id (str): The ID of the event to retrieve metadata for.

    Returns:
        dict: A dictionary containing event metadata:
            - summary (str): The summary of the event.
            - start (str): The start time of the event.
            - end (str): The end time of the event.
            - description (str): The description of the event.
    """
    event = calendar_service.events().get(calendarId=calendar_id, eventId=event_id).execute()
    return {
        'summary': event.get('summary'),
        'start': event.get('start').get('dateTime'),
        'end': event.get('end').get('dateTime'),
        'description': event.get('description')
    }

def get_meeting_transcripts(drive_service, file_id=''):
    """
    Retrieve and download a transcript file from Google Drive.

    Args:
        drive_service (googleapiclient.discovery.Resource): Google Drive API service object.
        file_id (str): The ID of the file to download.

    Returns:
        str: The filename where the transcript was saved.
    """
    request = drive_service.files().get_media(fileId=file_id)
    transcript_filename = 'transcript.txt'
    
    # Download the file from Google Drive
    with open(transcript_filename, 'wb') as file:
        downloader = googleapiclient.http.MediaIoBaseDownload(file, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            print(f"Download {int(status.progress() * 100)}%.")
    
    return transcript_filename

In [6]:
def main():
    """
    Main function to execute the workflow:
    - Authenticate Google services
    - Retrieve meeting metadata from Google Calendar
    - Download meeting transcript from Google Drive
    - Print out the metadata and the saved transcript file name
    """
    # Authenticate and get services
    calendar_service, drive_service = authenticate_google_services()
    
    # Example: Replace with actual event ID and file ID
    event_id = 'your_event_id'
    file_id = 'your_file_id'
    
    # Get meeting metadata
    metadata = get_meeting_metadata(calendar_service, event_id=event_id)
    print("Meeting Metadata:")
    print(metadata)
    
    # Get meeting transcript
    transcript_file = get_meeting_transcripts(drive_service, file_id=file_id)
    print(f"Transcript saved to {transcript_file}")

if __name__ == '__main__':
    main()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=173570917134-22hknipd8pbvl43hkik0287vii15au7k.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49752%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar.readonly+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=0MJ9Nuyu1X2Uykun4CfhANNyT1if16&access_type=offline


Run unit tests with:

python -m unittest discover -s tests

This is to test if everything is functionally working as expected.